In [ ]:
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np

from mdx2.io import loadobj


# Scaling model

Visualize the mdx2 scaling model.

## Parameters

In [ ]:
# These metadata fields are injected by mdx2.report, regardless of the template. Don't change this line.
_metadata: dict = {}

# Notebook parameters
scaling_model_sources: list[str] = None  # ScalingModel source encoded as a string nexus_file_path:object_name
offset_model_sources: list[str] = None  # OffsetModel sources encoded as a string nexus_file_path:object_name
absorption_model_sources: list[str] = None  # AbsorptionModel sources encoded as a string nexus_file_path:object_name
detector_model_sources: list[str] = None  # DetectorModel sources encoded as a string nexus_file_path:object_name

In [ ]:
# pretty-print the metadata and parameters
pprint({
    "metadata": _metadata,
    "parameters": {"scaling_model_sources": scaling_model_sources,
                   "offset_model_sources": offset_model_sources,
                   "absorption_model_sources": absorption_model_sources,
                   "detector_model_sources": detector_model_sources,
    },
})

## Results

### Scale

In [ ]:
for scaling_model_source in scaling_model_sources or []:
    try:
        scaling_model = loadobj(*scaling_model_source.split(":"))
    except Exception as e:
        print(f"failed to load {scaling_model_source}: {e}")
        continue
    b = scaling_model.to_xarray()
    b.plot(aspect=1.5, size=3)
    plt.title(f"{scaling_model_source}")

### Offset

In [ ]:
for offset_model_source in offset_model_sources or []:
    try:
        offset_model = loadobj(*offset_model_source.split(":"))
    except Exception as e:
        print(f"failed to load {offset_model_source}: {e}")
        continue
    c = offset_model.to_xarray()
    c.plot(aspect=1.5, size=3)
    plt.title(f"{offset_model_source}")

### Absorption

In [ ]:
for absorption_model_source in absorption_model_sources or []:
    try:
        absorption_model = loadobj(*absorption_model_source.split(":"))
    except Exception as e:
        print(f"failed to load {absorption_model_source}: {e}")
        continue
    a = absorption_model.to_xarray()
    phi_min = a.phi[0].data
    phi_max = a.phi[-1].data
    phi_interval = 360/8
    npts = int(1 + (phi_max - phi_min) / phi_interval)
    col_wrap = min(8, npts)
    a_sliced = a.sel(phi=np.linspace(phi_min, phi_max, npts),method='nearest')
    g = a_sliced.plot(
        yincrease=False,
        x='ix',
        y='iy',
        col='phi',
        cmap='coolwarm',
        robust=True,
        col_wrap=col_wrap,
        size=1.5,
        )
    for ax in g.axs.flatten():
        ax.set_aspect('equal')
    plt.title(f"{absorption_model_source}")

### Detector

In [ ]:
unique_detector_models = []
for detector_model_source in detector_model_sources or []:
    try:
        detector_model = loadobj(*detector_model_source.split(":"))
    except Exception as e:
        print(f"failed to load {detector_model_source}: {e}")
        continue
    # first, check if this detector model is identical to any of the previously loaded ones
    if any(np.array_equal(detector_model.u, d.u) for d in unique_detector_models):
        print(f"skipping duplicate detector model {detector_model_source}")
        continue
    unique_detector_models.append(detector_model)
    d = detector_model.to_xarray()
    d.plot(x='ix', y='iy', yincrease=False,cmap='coolwarm', robust=True, size=6)
    plt.title(f"{detector_model_source}")
